# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# Build the review queue. Every page is scored OUT-OF-FOLD (GroupKFold by client),
# so no page is ever ranked by a model that had seen its own client in training.
import os, json
from pathlib import Path
import numpy as np, pandas as pd, sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

SEED, VISIBLE_MIN, WEEKLY_CAPACITY = 42, 500, 50
if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days",
       "days_since_last_update", "ctr", "avg_position", "engagement_rate",
       "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent", "age_tier",
       "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
assert not ({"trend_direction", "trend_pct", "impressions_last_30d",
             "impressions_prev_30d"} & set(NUM + CAT)), "LEAK"

model = Pipeline([("prep", ColumnTransformer([
    ("n", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), NUM),
    ("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                    ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CAT)])),
    ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                 class_weight="balanced", n_jobs=-1, random_state=SEED))])

y = df["is_declining_label"].values
df["risk_score"] = cross_val_predict(model, df[NUM + CAT], y, cv=GroupKFold(n_splits=5),
                                     groups=df["client_id"], method="predict_proba")[:, 1]

# --- archetype -> action -> reason code. One archetype per page, first match wins. ---
df["visible"] = df["impressions_90d"] >= VISIBLE_MIN
df["has_position"] = df["avg_position"] > 0            # 0 = no data, not rank zero
df["pos_band"] = pd.cut(df["avg_position"].where(df["has_position"]),
                        [0, 10, 20, 1000], labels=["1-10", "11-20", "21+"])
df["ctr_below_band"] = df["ctr"] < df.groupby("pos_band", observed=True)["ctr"].transform("median")

def classify(r):
    if not r.visible:
        return ("low_visibility", "NO ACTION", "below the visibility floor of 500 impressions/90d")
    if r.has_position and r.avg_position <= 20 and r.ctr_below_band:
        return ("page_one_underperformer", "FIX CTR", "ranks 1-20 but CTR sits below its position band")
    if r.freshness_tier == "91-180":
        return ("stale_but_visible", "REFRESH", "not updated in 91-180 days and still earning impressions")
    if r.has_position and 10 < r.avg_position <= 20:
        return ("striking_distance", "SUPPORT", "just outside page 1 with competitive CTR")
    if r.has_position and r.avg_position > 20:
        return ("deep_and_visible", "MONITOR", "ranks past position 20 - CTR logic does not apply here")
    return ("visible_no_flag", "MONITOR", "visible, but no archetype flag fired")

df[["archetype", "action", "reason_code"]] = df.apply(classify, axis=1, result_type="expand")

queue = df[df["visible"]].sort_values("risk_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
queue["confidence"] = pd.cut(queue["risk_score"], [0, .5, .7, 1.01],
                             labels=["low", "medium", "high"], right=False)

def precision_at_k(s, labels, k):
    return float(np.asarray(labels)[np.argsort(-np.asarray(s))[:k]].mean())

QUEUE_BASE = queue["is_declining_label"].mean()
p_at = {k: precision_at_k(queue["risk_score"], queue["is_declining_label"], k) for k in (10, 50, 100)}
print(f"queue: {len(queue):,} reviewable pages of {len(df):,} | in-queue base rate {QUEUE_BASE:.3f}")
for k, v in p_at.items():
    print(f"  out-of-fold precision@{k:<4} {v:.3f}")

print("\nARCHETYPE -> ACTION (queue only)")
print(queue.groupby(["archetype", "action"], observed=True)
      .agg(pages=("rank", "size"), observed_decline_rate=("is_declining_label", "mean"))
      .sort_values("pages", ascending=False).round(3).to_string())

print("\nCONFIDENCE BANDS (from the out-of-fold probability)")
print(queue["confidence"].value_counts().reindex(["high", "medium", "low"]).to_string())

print("\nTOP 10 OF THE QUEUE")
print(queue.head(10)[["rank", "action", "archetype", "confidence", "impressions_90d",
                      "avg_position", "ctr"]].to_string(index=False))


queue: 16,726 reviewable pages of 30,000 | in-queue base rate 0.596
  out-of-fold precision@10   0.700
  out-of-fold precision@50   0.840
  out-of-fold precision@100  0.850

ARCHETYPE -> ACTION (queue only)
                                 pages  observed_decline_rate
archetype               action                               
stale_but_visible       REFRESH   5081                  0.579
page_one_underperformer FIX CTR   3673                  0.701
visible_no_flag         MONITOR   3447                  0.531
deep_and_visible        MONITOR   2485                  0.571
striking_distance       SUPPORT   2040                  0.584

CONFIDENCE BANDS (from the out-of-fold probability)
confidence
high      2745
medium    7958
low       6023

TOP 10 OF THE QUEUE
 rank  action               archetype confidence  impressions_90d  avg_position  ctr
    1 MONITOR        deep_and_visible       high             2877          23.2 0.03
    2 MONITOR        deep_and_visible       high           

### How to read the queue

Two things decide a row, and they do different jobs:

- **`risk_score`** (the model) decides **the order** — which page an editor opens first.
- **`archetype`** (a readable rule) decides **the action** — what they do when they get there.

They are deliberately separate. A probability cannot tell anyone what to type; a rule cannot rank 16,726 pages well. Splitting them means every row carries a reason a human can argue with, which is the only way a ranked list gets trusted.

### Archetype → action mapping

| Archetype | Action | Reason code shown to the editor | Pages | Observed decline rate |
|---|---|---|---|---|
| `page_one_underperformer` | **FIX CTR** | ranks 1–20 but CTR sits below its position band | 3,673 | 0.701 |
| `stale_but_visible` | **REFRESH** | not updated in 91–180 days and still earning impressions | 5,081 | 0.579 |
| `striking_distance` | **SUPPORT** | just outside page 1 with competitive CTR | 2,040 | 0.584 |
| `deep_and_visible` | **MONITOR** | ranks past position 20 — CTR logic does not apply here | 2,485 | 0.571 |
| `visible_no_flag` | **MONITOR** | visible, but no archetype flag fired | 3,447 | 0.531 |
| `low_visibility` | **NO ACTION** | below the visibility floor of 500 impressions/90d | (excluded) | 0.475 |

Two of these encode measured negatives rather than positives. `deep_and_visible` exists **because** the CTR-vs-position signal reversed past position 20 in my ML-07 audit (below-band CTR declined *less* there, −0.06), so those pages are deliberately not given a FIX CTR action. And `visible_no_flag` is an honest "no archetype fits" bucket rather than a forced label.

### The decay / refresh insight

Observed decline rate by freshness tier, across all 30,000 pages (base rate 0.542):

| Freshness | Pages | Declining |
|---|---|---|
| 0–30 days | 20,480 | 0.511 |
| 31–90 days | 175 | 0.589 |
| **91–180 days** | **9,171** | **0.611** |
| 181+ days | 174 | 0.471 |

**What I take from this:** in this snapshot, pages 91–180 days past their last update showed the highest decline rate — about 10 points above pages updated within 30 days. That window is where the REFRESH action is pointed, and it is the only freshness band besides `0-30` with enough pages to lean on.

**What I do not take from this:** that refreshing a page *causes* recovery. This is one 90-day cross-section with no intervention — I observed an association between staleness and decline, nothing more. The two middle and late tiers carry n=175 and n=174, and the stalest bucket actually reverses (0.471, the lowest of all four). I report that reversal rather than smoothing it, and I do not build an action on it.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

**Who:** a content/SEO editor or content lead working across client sites.

**For what:** deciding **which pages to open first** in a weekly review, and arriving with a starting hypothesis about what to look at on each one. It is a **reading order for a backlog nobody can clear**, not a verdict on any page.

**The scale problem it exists to solve.** 9,961 pages in this snapshot are both declining and still visible. At 50 reviews a week that is roughly **199 weeks of queue**. Nobody clears that list, so the only lever available is the order.

### Cost and value, in the units the user actually spends

| | |
|---|---|
| Reviewer capacity assumed | 50 pages/week |
| Random ordering would surface | ~30 genuinely declining pages per 50 |
| This queue surfaces | ~42 genuinely declining pages per 50 |
| **Net gain** | **~12 better-targeted reviews per editor-week** |
| Cost of a wrong pick | ~1–3 editor hours, plus the risk of editing a page that was fine |
| Cost of a missed page | a quarter of compounding decline on a page that still had traffic |

Twelve pages a week is the honest size of this. It is a **meaningful scheduling improvement, not a transformation** — and framing it that way is deliberate, because the failure mode for a tool like this is being sold as more than it is and then quietly abandoned.

The two error costs are asymmetric: a false positive is visible and recoverable (an editor opens a healthy page and moves on), a false negative is invisible (nobody learns what the queue never showed them). That asymmetry is why Section 3 requires a human at the point of action and why Section 4 monitors misses, not just hits.

### Where it stops being valid

- **Outside this portfolio.** 32 pseudonymised clients, one 90-day window. Nothing here establishes that the ordering transfers to a different client mix or a different quarter.
- **Below the visibility floor.** Pages under 500 impressions/90d are excluded, not ranked low. The queue has no opinion about them.
- **On the biggest pages.** In ML-09 the model caught **0 of 259** declining pages in the `excellent` impression tier. On the highest-traffic pages this queue is close to useless, and those are exactly the pages a client would ask about first.
- **On new pages.** The most confident false positives were pages with impressions and zero clicks that were actually *rising*. One snapshot cannot separate "fading" from "just arrived".
- **As a measure of quality.** `is_declining_label` is an impressions-trend bucket. A declining page is not a bad page, and a growing page is not a good one.
- **After the window moves.** The scores are computed from a fixed 90-day window. Once that window rolls, the queue is stale — see the retrain triggers in Section 4.

### The honest headline

> **Observed:** in a 90-day anonymised snapshot of 30,000 pages across 32 pseudonymised clients, an out-of-fold ranked queue placed genuinely declining pages in its top 50 at roughly 0.84, against an in-queue base rate of 0.60.
>
> **Directional:** under repeated client-grouped splits the model ordering beat the hand-written rule in 7 of 8 splits (ML-08), so the improvement is unlikely to be a single lucky split.
>
> **Decision-support:** these pages look worth reviewing first, and each row says why. Nothing here shows that acting on them will recover traffic — that would need pages assigned to review independently of how they looked, which no one did.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### The review rule, in one line

> **The queue chooses the order. A person chooses the action. Nothing ships without that person.**

### What a reviewer must check before acting on a row

Four checks, each one aimed at a failure I actually measured rather than imagined:

1. **Is this page new rather than fading?** Check whether it has a click history at all. The most confident false positives in ML-09 were zero-click pages that were genuinely *rising* — the model cannot tell "just arrived" from "on the way out". *If the page is young and climbing, drop it from the queue and move on.*
2. **Is the CTR problem real, or structural?** A very large page ranking 1–3 with low CTR is often answering the query in the search result itself. That is not a defect to fix. *Read the intent before rewriting a title.*
3. **Does the action match what you see?** The archetype is a rule, not a diagnosis. If a `REFRESH` row is plainly a CTR problem, do the CTR work and note the mismatch — those notes are the training data for the next version of the rules.
4. **Would an edit risk what the page already earns?** A rewrite that shifts intent can lose impressions the page currently has. On high-traffic pages the safe move is a small, reversible change.

**Confidence bands set the depth of review, not whether to review.** `high` (score ≥ 0.70, 2,745 pages) — the pattern is clear, standard review. `medium` (0.50–0.70, 7,958) — expect a higher miss rate. `low` (< 0.50, 6,023) — treat as unranked; these are in the queue only because they cleared the visibility floor.

---

### The no-go list — what must never be automated

**1. No automated content changes.** The system may not write, rewrite, or publish titles, meta descriptions, or body copy. It ranks; a person edits. An unreviewed rewrite driven by a 0.84-precision ranking will damage pages at a predictable rate.

**2. No automated pruning or deletion.** Nothing in this work identifies a page as worthless. `low_visibility` means "not measurable here", never "delete". The label is an impressions trend, and a low-traffic page can carry real strategic or legal weight that this dataset cannot see.

**3. No client-facing forecasts.** No row of this queue may be reported as "this page will decline". The model ranks pages by resemblance to pages that *were* declining in one past window. "Predicted to decline" is a sentence the evidence cannot carry.

**4. No automatic prioritisation for the biggest pages.** The measured blind spot — 0 of 259 declining pages caught in the `excellent` tier — means high-traffic pages need a human-led process, not this queue. Routing them through it would create false comfort.

**5. No performance management.** These scores must never be used to evaluate the writers or agencies who produced the pages. The label reflects search-demand movement, not effort or craft, and using it that way would be both wrong and unfair.

**6. No silent scoring of new clients.** A client whose pages were not in the training mix gets a queue reviewed by a human before it is used. ML-09 measured one held-out client scoring 0.445 — below a coin flip. That can happen to any new client, and nothing in the score announces it.

**7. No auto-refresh scheduling from the decay finding.** The 91–180 day window is an observed association in one snapshot. Wiring it into a calendar job would convert a correlation into a policy without anyone deciding to.

### Why this list is this strict

The honest gain here is roughly twelve better-targeted reviews per editor-week. That is worth having, and it is nowhere near enough to justify removing the human. Every automation above would trade a small scheduling gain for a class of error that nobody would notice until a client did.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [2]:
# Monitoring thresholds derived from THIS run, so the trip-wires are calibrated
# to measured spread rather than to round numbers someone liked.
SPLIT_SD = 0.108          # P@50 spread across 8 client-grouped splits (ML-08)
BASELINE_P50 = 0.605      # the frozen ML-07 rule, same splits (ML-08)

current = {
    "queue_p50": p_at[50],
    "queue_base_rate": QUEUE_BASE,
    "portfolio_base_rate": df["is_declining_label"].mean(),
    "share_high_confidence": float((queue["confidence"] == "high").mean()),
    "median_days_with_impressions": float(df["days_with_impressions"].median()),
    "n_clients": int(df["client_id"].nunique()),
}

triggers = {
    "RETRAIN - queue precision decayed": {
        "watch": "precision@50 on a human-audited sample of the shipped queue",
        "trip_if": f"< {BASELINE_P50:.2f}",
        "why": "that is the hand-rule's score. Below it the model has stopped earning its place.",
    },
    "INVESTIGATE - precision moved but may be noise": {
        "watch": "precision@50, same audit",
        "trip_if": f"drops more than {2*SPLIT_SD:.2f} below {current['queue_p50']:.2f}",
        "why": "2 sd of the measured split-to-split spread; smaller moves are not signal.",
    },
    "RETRAIN - label drift": {
        "watch": "portfolio share of pages labelled declining",
        "trip_if": f"outside {current['portfolio_base_rate']-0.10:.2f}-{current['portfolio_base_rate']+0.10:.2f}",
        "why": "the base rate IS the bar; if it moves, every precision number changes meaning.",
    },
    "REVIEW - population drift": {
        "watch": "median days_with_impressions across the portfolio",
        "trip_if": f"moves more than 20% from {current['median_days_with_impressions']:.0f}",
        "why": "the model's strongest feature (ML-08 permutation importance). If its distribution shifts, the ranking rests on different ground.",
    },
    "REVIEW - new clients entered": {
        "watch": "client count and the share of queue rows from unseen clients",
        "trip_if": f"> 10% of queue rows come from clients absent at training (now {current['n_clients']} clients)",
        "why": "one held-out client scored 0.445 in ML-09. New clients are the known failure mode.",
    },
    "REBUILD - window rolled": {
        "watch": "age of the 90-day snapshot the scores were computed from",
        "trip_if": "> 30 days old",
        "why": "scores describe a fixed window; a third of it is stale after a month.",
    },
}

print("MONITORING / RETRAIN TRIGGERS")
print("=" * 78)
for name, t in triggers.items():
    print(f"\n{name}")
    print(f"   watch   : {t['watch']}")
    print(f"   trip if : {t['trip_if']}")
    print(f"   why     : {t['why']}")

print("\n" + "=" * 78)
print("CURRENT VALUES (the reference point every trigger is measured against)")
print("=" * 78)
for k, v in current.items():
    print(f"  {k:<32} {v:.3f}" if isinstance(v, float) else f"  {k:<32} {v}")

print("\nCadence: audit 20 sampled queue rows monthly; retrain quarterly or on any")
print("RETRAIN trigger, whichever comes first. Retraining is cheap (~2 min); the")
print("expensive part is the human audit, so that is what the cadence is built around.")


MONITORING / RETRAIN TRIGGERS

RETRAIN - queue precision decayed
   watch   : precision@50 on a human-audited sample of the shipped queue
   trip if : < 0.60
   why     : that is the hand-rule's score. Below it the model has stopped earning its place.

INVESTIGATE - precision moved but may be noise
   watch   : precision@50, same audit
   trip if : drops more than 0.22 below 0.84
   why     : 2 sd of the measured split-to-split spread; smaller moves are not signal.

RETRAIN - label drift
   watch   : portfolio share of pages labelled declining
   trip if : outside 0.44-0.64
   why     : the base rate IS the bar; if it moves, every precision number changes meaning.

REVIEW - population drift
   watch   : median days_with_impressions across the portfolio
   trip if : moves more than 20% from 81
   why     : the model's strongest feature (ML-08 permutation importance). If its distribution shifts, the ranking rests on different ground.

REVIEW - new clients entered
   watch   : client coun

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("work/outputs"); OUT.mkdir(parents=True, exist_ok=True)
FIG = Path("work/figures"); FIG.mkdir(parents=True, exist_ok=True)

# Palette: validated with the dataviz palette checker (light surface #fcfcfb) --
# all six checks pass, worst adjacent CVD dE 24.7, normal-vision dE 33.6.
BLUE, ORANGE, SURFACE = "#2a78d6", "#eb6834", "#fcfcfb"
PALE, INK, MUTED = "#b9d3f2", "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
                     "font.size": 10, "axes.edgecolor": "#d8d7d2",
                     "text.color": INK, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED})

def strip(ax):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0)

# --- FIGURE 1: what the ranking is worth, with its spread ---------------------
methods = ["Base rate\n(random order)", "Hand rule\n(ML-07)",
           "Logistic\nregression", "Random forest\n(shipped)"]
vals = [0.495, 0.605, 0.735, 0.805]
sds = [0.068, 0.114, 0.120, 0.108]
colors = [MUTED, ORANGE, BLUE, BLUE]

fig, ax = plt.subplots(figsize=(7.6, 4.2))
bars = ax.bar(methods, vals, width=0.62, color=colors, zorder=3,
              edgecolor=SURFACE, linewidth=2)            # 2px surface gap between fills
ax.errorbar(methods, vals, yerr=sds, fmt="none", ecolor=MUTED,
            elinewidth=1.2, capsize=5, zorder=4)
# Values sit INSIDE the bars -- the error bars own the space above them.
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v - 0.06, f"{v:.2f}",
            ha="center", fontsize=12, fontweight="bold", color="#ffffff")
ax.axhline(0.495, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=2)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Precision@50 (held-out clients)")
ax.set_title("Ranking a review queue, measured against the floor",
             fontsize=12.5, fontweight="bold", loc="left", pad=12)
ax.yaxis.grid(True, color="#eceae4", lw=1, zorder=0); strip(ax)
fig.text(0.01, -0.03, "8 client-grouped splits; error bars = 1 sd. The dashed line is the base "
                      "rate. Bars whose error bars overlap are not distinguishable here.",
         fontsize=8, color=MUTED)
fig.tight_layout()
fig.savefig(FIG / "fig1_queue_vs_baseline.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
plt.close(fig)

# --- FIGURE 2: the decay / refresh insight, with sample size made visible -----
tiers = ["0-30", "31-90", "91-180", "181+"]
f = (df.groupby("freshness_tier", observed=True)["is_declining_label"]
       .agg(n="size", rate="mean").reindex(tiers))
f["n"] = f["n"].astype(int)
small = f["n"] < 1000                                     # texture = "do not lean on this"

fig, ax = plt.subplots(figsize=(7.6, 4.2))
bars = ax.bar(tiers, f["rate"], width=0.62, zorder=3, edgecolor=SURFACE, linewidth=2,
              color=[(PALE if s else BLUE) for s in small],
              hatch=["///" if s else "" for s in small])
# Labels inside the bars, so nothing collides with the base-rate line.
for b, (_, row), is_small in zip(bars, f.iterrows(), small):
    on_bar = INK if is_small else "#ffffff"
    ax.text(b.get_x() + b.get_width() / 2, row["rate"] - 0.045, f"{row['rate']:.3f}",
            ha="center", fontsize=12, fontweight="bold", color=on_bar)
    ax.text(b.get_x() + b.get_width() / 2, 0.022, f"n = {int(row['n']):,}",
            ha="center", fontsize=9, color=on_bar)
BASE = df["is_declining_label"].mean()
ax.axhline(BASE, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)  # behind bars: no label collisions
ax.text(-0.44, BASE + 0.012, f"portfolio base rate {BASE:.3f}",
        ha="left", fontsize=8.5, color=MUTED)
ax.set_ylim(0, 0.72)
ax.set_ylabel("Share of pages observed declining")
ax.set_xlabel("Days since last update")
ax.set_title("Decline peaks 91-180 days after the last update",
             fontsize=12.5, fontweight="bold", loc="left", pad=12)
ax.yaxis.grid(True, color="#eceae4", lw=1, zorder=0); strip(ax)
fig.text(0.01, -0.03, "Hatched bars carry n < 1,000 and are not reliable -- note the stalest "
                      "tier reverses. Observed association in one 90-day snapshot; not evidence "
                      "that refreshing causes recovery.", fontsize=8, color=MUTED)
fig.tight_layout()
fig.savefig(FIG / "fig2_decay_by_freshness.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
plt.close(fig)

# --- the queue CSV (gitignored by design; regenerates on every run) -----------
cols = ["rank", "content_id", "client_id", "risk_score", "confidence", "action",
        "archetype", "reason_code", "impressions_90d", "avg_position", "ctr",
        "days_since_last_update", "freshness_tier"]
queue[cols].to_csv(OUT / "content_action_playbook_queue.csv", index=False)

# --- the receipts JSON (committed; the paper's numbers trace back to this) ----
metrics = {
    "seed": SEED, "sklearn_version": sklearn.__version__,
    "scoring": "out-of-fold, GroupKFold(5) by client_id",
    "visibility_floor_impressions_90d": VISIBLE_MIN,
    "weekly_capacity_assumed": WEEKLY_CAPACITY,
    "queue_rows": int(len(queue)), "portfolio_rows": int(len(df)),
    "queue_base_rate": round(float(QUEUE_BASE), 4),
    "portfolio_base_rate": round(float(BASE), 4),
    "out_of_fold_precision_at_k": {str(k): round(v, 4) for k, v in p_at.items()},
    "validated_comparison_ml08": {"base_rate": 0.495, "hand_rule": 0.605,
                                  "logistic_regression": 0.735, "random_forest": 0.805,
                                  "p50_sd": SPLIT_SD, "rf_wins_vs_rule": "7/8 splits"},
    "confidence_counts": {str(k): int(v) for k, v in queue["confidence"].value_counts().items()},
    "archetype_action_counts": {a: {"action": g["action"].iloc[0], "pages": int(len(g))}
                                for a, g in queue.groupby("archetype", observed=True)},
    "decay_by_freshness": {k: {"n": int(v["n"]), "decline_rate": round(float(v["rate"]), 4)}
                           for k, v in f.iterrows()},
    "value_per_editor_week": {
        "capacity": WEEKLY_CAPACITY,
        "expected_declining_random_order": round(QUEUE_BASE * WEEKLY_CAPACITY, 1),
        "expected_declining_this_queue": round(p_at[50] * WEEKLY_CAPACITY, 1),
        "net_gain_pages": round((p_at[50] - QUEUE_BASE) * WEEKLY_CAPACITY, 1)},
    "monitoring_triggers": triggers,
    "not_automated": ["content edits", "pruning/deletion", "client-facing forecasts",
                      "high-traffic prioritisation", "performance management",
                      "silent scoring of new clients", "auto-refresh scheduling"],
}
(OUT / "playbook_metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

print("EXPORTS FOR THE PAPER")
print(f"  {OUT/'content_action_playbook_queue.csv'}   {len(queue):,} rows   [gitignored by design]")
print(f"  {OUT/'playbook_metrics.json'}   [committed - the paper's receipts]")
for p in sorted(FIG.glob("fig*")):
    print(f"  {p}   [committed - reused in the paper]")


EXPORTS FOR THE PAPER
  work\outputs\content_action_playbook_queue.csv   16,726 rows   [gitignored by design]
  work\outputs\playbook_metrics.json   [committed - the paper's receipts]
  work\figures\fig1_queue_vs_baseline.png   [committed - reused in the paper]
  work\figures\fig2_decay_by_freshness.png   [committed - reused in the paper]


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — pseudonymous IDs only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What this notebook hands to next week's paper

| File | Role | In git? |
|---|---|---|
| `work/outputs/content_action_playbook_queue.csv` | The ranked queue, 16,726 rows | **No** — gitignored by design, regenerates on every run |
| `work/outputs/playbook_metrics.json` | Every number the paper will quote, with seeds and library versions | **Yes** — the receipts |
| `work/figures/fig1_queue_vs_baseline.png` | The headline comparison, with error bars | **Yes** |
| `work/figures/fig2_decay_by_freshness.png` | The decay/refresh insight, with n on every bar | **Yes** |

Both figures were built with a palette checked by the dataviz validator (all six checks pass on the light surface), and both carry their caveat *inside the figure* rather than in a caption someone can crop off: figure 1 shows the split-to-split spread as error bars, and figure 2 hatches every bar with n < 1,000.

**This is a plan, not a deployment.** Nothing here runs on a schedule, writes to a CMS, or reaches a client unreviewed. The playbook's whole claim is that it is a better reading order for a backlog — worth roughly twelve better-targeted reviews per editor-week — with a person deciding every action.
